In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, ElasticNet, Lasso, Ridge
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import PolynomialFeatures
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score

from life_saving_tools.Notification import Notification

In [10]:
X = pd.read_csv('data/train_final.csv')
test = pd.read_csv('data/test_final.csv')
y = pd.read_csv('data/y.csv')

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.12, random_state=42)
len(X_train), len(X_test)

(1284, 176)

### Base Model

In [17]:
n = Notification()

In [37]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [38]:
y_pred = lr.predict(X_test)
mean_squared_error(y_test, y_pred)**0.5

36165.77423028959

In [39]:
y_pred = lr.predict(X_train)
mean_squared_error(y_train, y_pred)**0.5

28531.377331518488

Linear model is of no use! Let's try a polynomial model of order 2.

In [40]:
pf = PolynomialFeatures(degree=2)
X_train_pf = pf.fit_transform(X_train)
X_test_pf = pf.fit_transform(X_test)

In [41]:
lr_pf = LinearRegression()
lr_pf.fit(X_train_pf, y_train)

y_pred = lr_pf.predict(X_test_pf)
msre = mean_squared_error(y_test, y_pred)**0.5
text = "The mean squared error is {}".format(msre)
print(text)

The mean squared error is 72013.23219765784


In [42]:
y_pred_test = lr_pf.predict(X_train_pf)
msre_test = mean_squared_error(y_train, y_pred_test)**0.5
text = "The mean squared error is {}".format(msre_test)
print(text)

The mean squared error is 163.78743537750125


Okay, the model is overfitting, severly. Let's try Ridge, Lasso, and Elastic Net.

In [59]:
def train_model(model, data, test_data, text = False):
    model.fit(data[0], data[1])
    y_pred = model.predict(test_data[0])
    msre_test = mean_squared_error(y_pred, test_data[1])**0.5
    y_pred_train = model.predict(data[0])
    msre_train = mean_squared_error(y_pred_train, data[1])**0.5
    text = f"The mean squared error for test data is {msre_test}"
    text+= f"\nThe mean squared error for train data is {msre_train}"
    print(text)
    if text:
        n.send_whatsapp_text(text)
    return model

In [55]:
rr = train_model(Ridge(), (X_train_pf, y_train), (X_test_pf, y_test))

The mean squared error for test data is 48458.56238434578
The mean squared error for train data is 2041.2016774418073


In [57]:
grid = {'alpha': np.logspace(-2, 2, 20)}
rm = Ridge()
kgrid = GridSearchCV(rm, grid, cv=3, scoring='neg_mean_squared_error', verbose = 2)
kgrid.fit(X_train_pf, y_train)
best_model = kgrid.best_estimator_
pred = best_model.predict(X_test_pf)
msre = mean_squared_error(y_test, pred)**0.5
text = "The mean squared error is {}".format(msre)
print(text)
# n.send_whatsapp_text(text)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] alpha=0.01 ......................................................


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV] ....................................... alpha=0.01, total=   0.3s
[CV] alpha=0.01 ......................................................


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s remaining:    0.0s


[CV] ....................................... alpha=0.01, total=   0.3s
[CV] alpha=0.01 ......................................................
[CV] ....................................... alpha=0.01, total=   0.3s
[CV] alpha=0.016237767391887217 ......................................
[CV] ....................... alpha=0.016237767391887217, total=   0.3s
[CV] alpha=0.016237767391887217 ......................................
[CV] ....................... alpha=0.016237767391887217, total=   0.3s
[CV] alpha=0.016237767391887217 ......................................
[CV] ....................... alpha=0.016237767391887217, total=   0.3s
[CV] alpha=0.026366508987303583 ......................................
[CV] ....................... alpha=0.026366508987303583, total=   0.3s
[CV] alpha=0.026366508987303583 ......................................
[CV] ....................... alpha=0.026366508987303583, total=   0.3s
[CV] alpha=0.026366508987303583 ......................................
[CV] .

[Parallel(n_jobs=1)]: Done  60 out of  60 | elapsed:   20.6s finished


The mean squared error is 30310.981574056867


In [60]:
from xgboost import XGBRegressor
xgbr = XGBRegressor()
train_model(xgbr, (X_train_pf, y_train), (X_test_pf, y_test), text=True)

The mean squared error for test data is 30099.184082643445
The mean squared error for train data is 1540.167510861688
SMca78c4f8446145baa5fe86da5c58d415


XGBRegressor(base_score=0.5, booster='gbtree', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=1, enable_categorical=False,
             gamma=0, gpu_id=-1, importance_type=None,
             interaction_constraints='', learning_rate=0.300000012,
             max_delta_step=0, max_depth=6, min_child_weight=1, missing=nan,
             monotone_constraints='()', n_estimators=100, n_jobs=8,
             num_parallel_tree=1, predictor='auto', random_state=0, reg_alpha=0,
             reg_lambda=1, scale_pos_weight=1, subsample=1, tree_method='exact',
             validate_parameters=1, verbosity=None)

In [61]:
grid = {'max_depth': [4, 5, 6, 7, 8, ],
        'learning_rate': [0.1, 0.2, 0.3, 0.4, 0.5],
        "n_estimators": [50, 100, 150, 200, 300],
        "booster": ["gbtree", "gblinear", "dart"],
        "reg_alpha": [0, 0.1,  0.2],
        }
xgbr = XGBRegressor()
gridsearch = GridSearchCV(xgbr, grid, cv=3, scoring='neg_mean_squared_error', verbose = 2)
gridsearch.fit(X_train_pf, y_train)
best_model = gridsearch.best_estimator_
test_pred = best_model.predict(X_test_pf)
train_pred = best_model.predict(X_train_pf)
msre_test = mean_squared_error(y_test, test_pred)**0.5
msre_train = mean_squared_error(y_train, train_pred)**0.5
best_params = gridsearch.best_params_
text = f"Best params are {best_params}"
text += f"\nThe mean squared error for test data is {msre_test}"
text+= f"\nThe mean squared error for train data is {msre_train}"
print(text)
n.send_whatsapp_text(text)

Fitting 3 folds for each of 1125 candidates, totalling 3375 fits
[CV] booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0 


[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.


[CV]  booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0, total=   5.8s
[CV] booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0 


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    5.7s remaining:    0.0s


[CV]  booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0, total=   5.4s
[CV] booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0 
[CV]  booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0, total=   4.6s
[CV] booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0.1 
[CV]  booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0.1, total=   4.3s
[CV] booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0.1 
[CV]  booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0.1, total=   5.0s
[CV] booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0.1 
[CV]  booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0.1, total=   4.2s
[CV] booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=0.2 
[CV]  booster=gbtree, learning_rate=0.1, max_depth=4, n_estimators=50, reg_alpha=

KeyboardInterrupt: 